In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

Pdf To Chunks


In [2]:
path = "Generic Motion Transfer from Video to Static Image Using Deep Learning Techniques.pdf"

# Load PDF
loader = PyPDFLoader(path)
docs = loader.load()

print("Pages:", len(docs))

# Split into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True
)

texts = text_splitter.split_documents(docs)

print("Chunks:", len(texts))
# Inspect one chunk
print("\nSample Chunk:\n")
print(texts[1].page_content)
print("\nMetadata:\n", texts[1].metadata)

Pages: 14
Chunks: 48

Sample Chunk:

This paper presents a practical motion transfer system that animates a static portrait image by 
transferring head motion and facial expressions from a driving video. Built on the First Order 
Motion Model, the system extends it with a modular compositing pi peline addressing real -
world deployment challenges. The pipeline incorporates unsupervised keypoint detection, 
relative dense motion estimation, and Exponential Moving Average temporal smoothing to 
reduce inter-frame jitter. A combined masking strategy int ersects a convex hull mask derived 
from keypoint geometry with an occlusion-aware mask from the generator's confidence output, 
producing feathered boundaries for seamless blending. Geometric alignment via similarity 
transformation positions generated frame s correctly in the driving domain. Gradient -domain 
Poisson blending composites the animated face onto the driving background without visible

Metadata:
 {'producer': 'Microsoft® Wor

In [3]:
from langchain_ollama import OllamaEmbeddings
embeddings = OllamaEmbeddings(model="nomic-embed-text")

Chunks into vectors

In [4]:
vector1=embeddings.embed_query(texts[0].page_content)
vector2=embeddings.embed_query(texts[1].page_content)
assert len(vector1) == len(vector2)
print(f"generated vectors of length: {len(vector1)}")
print(vector1[:10])

generated vectors of length: 768
[-0.0030034187, 0.053001184, -0.17305836, -0.039758094, 0.05196827, -0.03162184, 0.07169817, 0.01377033, 0.005577626, -0.0038509406]


vectors -> vector DB

In [5]:
vector_store=Chroma(collection_name="pdf_collection", embedding_function=embeddings,persist_directory="./chroma_langchain_db")

In [6]:
ids=vector_store.add_documents(documents=texts)

In [7]:
res=vector_store.similarity_search_with_score("What is the main contribution of the paper?")
docs, scores = res[0]
print(f"Score: {scores}")
print(res[0])

Score: 0.8974055647850037
(Document(id='c8dd1091-c7e1-475d-ba5d-2d33765bcfc3', metadata={'creationdate': '2026-03-28T22:58:37+05:30', 'moddate': '2026-03-28T22:58:37+05:30', 'creator': 'Microsoft® Word 2021', 'start_index': 0, 'author': 'meghanakaparapu01@gmail.com', 'page_label': '14', 'total_pages': 14, 'source': 'Generic Motion Transfer from Video to Static Image Using Deep Learning Techniques.pdf', 'producer': 'Microsoft® Word 2021', 'page': 13}, page_content='in Proceedings of the IEEE/CVF Conference on Computer Vision and Pattern Recognition \n(CVPR), 2020, pp. 14558–14567. \n[18] J. Zhao, \n “Thin-Plate Spline Motion Model for Image Animation,” \n in Proceedings of the IEEE/CVF Conference on Computer Vision and Pattern Recognition \n(CVPR), 2022. \n[19] N. A. Teka et al., \n “AMT-Net: Adversarial Motion Transfer Network With Disentangled Shape and Pose for \nRealistic Image Animation,” \n IEEE Access, vol. 11, pp. 1–12, 2023.'), 0.8974055647850037)


In [8]:
from langchain.tools import tool

@tool(response_format="content_and_artifact")
def retrieve_content(query: str):
    """retrieving content"""
    retrieved_docs = vector_store.similarity_search(query, k=2)

    serialised = "\n\n".join(
        f"Source: {doc.metadata}\nContent: {doc.page_content}"
        for doc in retrieved_docs
    )

    return serialised, retrieved_docs

In [9]:
from langchain_ollama import ChatOllama
llm=ChatOllama(
    model="llama3",
    temperature=0,
    validate_model_on_init=True

)

In [ ]:
query = "What are the limitations and future work mentioned by the authors?"

# Retrieval
retrieved_docs = vector_store.similarity_search(query, k=3)

# Augmentation
context = "\n\n".join(doc.page_content for doc in retrieved_docs)

# Generation
prompt = f"""
Answer the question based ONLY on the context below.

Context:
{context}

Question:
{query}
"""
# Generate answer
response = llm.invoke(prompt)
print(response.content)

Based on the provided context, there is no mention of limitations or future work by the authors. The text only lists references to papers published in Proceedings of the IEEE/CVF Conference on Computer Vision and Pattern Recognition (CVPR) and IEEE Access. It does not contain any information about limitations or future work mentioned by the authors.
